In [2]:
# PHASE 8 — CUSTOMER DECISION LAYER
# Customer Retention Intelligence

import pandas as pd
import numpy as np

In [3]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()

engine = create_engine(os.getenv("DATABASE_URL"))

### 1. Load customer_360


In [4]:
df = pd.read_sql(
    "SELECT * FROM customer_360",
    engine
)

print("Loaded customer_360:", df.shape)

Loaded customer_360: (1200, 30)


### 2. Create business signals

In [5]:
# Missing support metrics = zero activity
df["ticket_count"] = df["ticket_count"].fillna(0)
df["high_priority_tickets"] = df["high_priority_tickets"].fillna(0)
df["urgent_tickets"] = df["urgent_tickets"].fillna(0)
df["unresolved_tickets"] = df["unresolved_tickets"].fillna(0)

# Financial metrics
df["mrr"] = df["mrr"].fillna(0)
df["open_amount"] = df["open_amount"].fillna(0)
df["uncollectible_amount"] = df["uncollectible_amount"].fillna(0)

### 3. Rule-based risk signals

In [6]:
# Low engagement
df["low_engagement"] = (
    df["user_count"] <= 5
).astype(int)

# Low commercial value
df["low_mrr"] = (
    df["mrr"] < df["mrr"].median()
).astype(int)

# Payment risk
df["payment_risk"] = (
    (df["open_amount"] > 0) |
    (df["uncollectible_amount"] > 0)
).astype(int)

# Support pressure
df["support_pressure"] = (
    (df["high_priority_tickets"] > 0) |
    (df["urgent_tickets"] > 0)
).astype(int)

# Unresolved support issues
df["unresolved_risk"] = (
    df["unresolved_tickets"] > 0
).astype(int)

# Low satisfaction
df["low_satisfaction"] = (
    df["avg_satisfaction"].fillna(
        df["avg_satisfaction"].median()
    ) < 3.5
).astype(int)

### 4. Customer Health Score


In [7]:
risk_columns = [
    "low_engagement",
    "low_mrr",
    "payment_risk",
    "support_pressure",
    "unresolved_risk",
    "low_satisfaction"
]

df["risk_points"] = df[risk_columns].sum(axis=1)

# 100 = healthiest
df["health_score"] = (
    100 -
    (df["risk_points"] / len(risk_columns) * 100)
).round(0)

### 5. Health segments

In [8]:
df["health_segment"] = pd.cut(
    df["health_score"],
    bins=[-1, 39, 59, 79, 100],
    labels=[
        "Critical",
        "At Risk",
        "Watch",
        "Healthy"
    ]
)

### 6. Revenue exposure 

In [9]:
# Existing churned revenue
df["churned_revenue"] = np.where(
    df["subscription_status"] == "churned",
    df["mrr"],0
)

# Active customers currently carrying risk signals
df["active_at_risk"] = (
    (df["subscription_status"] == "active") &
    (df["risk_points"] >= 2)
).astype(int)

df["active_revenue_at_risk"] = np.where(
    df["active_at_risk"] == 1,
    df["mrr"],0
)

### 7. Customer intelligence table

In [10]:
customer_intelligence = df[
    [
        "account_id",
        "company_name",
        "industry",
        "country",
        "plan",
        "subscription_status",
        "mrr",
        "seats",
        "user_count",
        "users_per_seat",
        "ticket_count",
        "avg_satisfaction",
        "avg_resolution_hours",
        "open_amount",
        "uncollectible_amount",
        "risk_points",
        "health_score",
        "health_segment",
        "active_at_risk",
        "active_revenue_at_risk"
    ]
].copy()

### 8. Push to PostgreSQL


In [11]:
customer_intelligence.to_sql(
    "customer_intelligence",
    engine,
    if_exists="replace",
    index=False
)

print("\ncustomer_intelligence created successfully.")


customer_intelligence created successfully.


### 9. Executive summary

In [12]:
print("\n==============================")
print("EXECUTIVE RETENTION SUMMARY")
print("==============================")

print(
    "\nCustomers:",
    len(customer_intelligence)
)

print(
    "Healthy:",
    (customer_intelligence["health_segment"] == "Healthy").sum()
)

print(
    "Watch:",
    (customer_intelligence["health_segment"] == "Watch").sum()
)

print(
    "At Risk:",
    (customer_intelligence["health_segment"] == "At Risk").sum()
)

print("Critical:",(customer_intelligence["health_segment"] == "Critical").sum())

print("\nActive customers:", (customer_intelligence["subscription_status"]== "active").sum())

print( "Active customers flagged at risk:",customer_intelligence["active_at_risk"].sum())

print("\nCurrent MRR:",round(customer_intelligence["mrr"].sum(),2))

print("Active MRR exposed to risk:",round(customer_intelligence["active_revenue_at_risk"].sum(),2))

print("MRR from historically churned customers:",round(customer_intelligence["mrr"][customer_intelligence["subscription_status"] == "churned"].sum(),2))


EXECUTIVE RETENTION SUMMARY

Customers: 1200
Healthy: 235
Watch: 338
At Risk: 309
Critical: 318

Active customers: 1030
Active customers flagged at risk: 822

Current MRR: 1020471.0
Active MRR exposed to risk: 607978.0
MRR from historically churned customers: 58530.0


### 10. Health segment summary

In [13]:
segment_summary = (
    customer_intelligence
    .groupby("health_segment", observed=True).agg(
        customers=("account_id", "count"),
        total_mrr=("mrr", "sum"),
        avg_mrr=("mrr", "mean"),
        avg_health_score=("health_score", "mean")
    ).sort_values("avg_health_score")
)

print("\n==============================")
print("HEALTH SEGMENTS")
print("==============================")

print(segment_summary.round(2).to_string())


HEALTH SEGMENTS
                customers  total_mrr  avg_mrr  avg_health_score
health_segment                                                 
Critical              318    35304.0   111.02             27.42
At Risk               309   187828.0   607.86             50.00
Watch                 338   459242.0  1358.70             67.00
Healthy               235   338097.0  1438.71             85.31


### 11. Highest-value active customers needing attention

In [15]:
priority_customers = (
    customer_intelligence[
        customer_intelligence["active_at_risk"] == 1
    ]
    .sort_values(
        ["mrr", "risk_points"],
        ascending=[False, False]
    )
    .head(20)
)

print("\n==================================")
print("TOP 20 ACTIVE CUSTOMERS FOR REVIEW")
print("==================================")

print(
    priority_customers[
        [
            "account_id",
            "company_name",
            "plan",
            "mrr",
            "risk_points",
            "health_score",
            "health_segment",
            "open_amount",
            "uncollectible_amount",
            "ticket_count",
            "avg_satisfaction"
        ]
    ].to_string(index=False)
)


TOP 20 ACTIVE CUSTOMERS FOR REVIEW
 account_id              company_name       plan     mrr  risk_points  health_score health_segment  open_amount  uncollectible_amount  ticket_count  avg_satisfaction
     457669      Unified Ops Partners Enterprise 25122.0            2          67.0          Watch         0.00                  0.00           2.0              1.00
     675202     NodeSync Collective 1 Enterprise 22673.0            2          67.0          Watch     23963.11                  0.00           6.0              4.33
     662747        ZeroFriction Group Enterprise 18012.0            3          50.0        At Risk     17859.40                  0.00           4.0              3.25
     757620    GridLogic Collective 1 Enterprise 15958.0            3          50.0        At Risk     15852.20              15680.98           5.0              3.20
     847537         Catalyst CRM Labs Enterprise 15484.0            2          67.0          Watch     16008.91                  0.00 